# Feature Analysis

How each feature actually relates to `Survived`, so feature engineering is driven by evidence rather than guesswork.

Cleaning logic lives in `preprocessing.ipynb` / `scripts/preprocessing.py`, and model training lives in `scripts/train.py` (`uv run train <config>`). This notebook imports the cleaning steps rather than repeating them, so there is only one definition of how the data gets prepared.

In [ ]:
import pandas as pd

# raw and uncleaned -- the shared cleaning steps get applied in the next cell
df = pd.read_csv("data/train.csv")
df.info()  # column types, missing values

## Split, then apply the shared cleaning steps

Splitting first keeps the analysis honest: every number below is computed from `train` alone, so the held-out rows never influence which features get engineered.

The cleaning steps come straight from `build_preprocessing_pipeline()`, sliced to stop before `add_interaction`. That leaves `Pclass` and `Embarked` as readable categories instead of one-hot columns, which both the charts and the statsmodels formula below need. Slicing by step *name* rather than a hardcoded number means reordering the pipeline can't silently change what runs here.

In [ ]:
from sklearn.model_selection import train_test_split

from preprocessing import build_preprocessing_pipeline

# the test split is deliberately unused -- it exists only so the analysis never sees those rows
train_raw, _ = train_test_split(
    df, test_size=0.2, stratify=df["Survived"], random_state=42
)

full = build_preprocessing_pipeline()
step_names = [name for name, _ in full.steps]
# everything before add_interaction: drop columns, encode Sex, impute Age and Embarked
cleaning = full[: step_names.index("add_interaction")]

train = cleaning.fit_transform(train_raw)
train.info()  # confirm: no nulls in Age/Embarked, Pclass still 1/2/3

In [ ]:
import matplotlib.pyplot as plt

import seaborn as sns

## Survival rate by Sex



`Survived` is 0/1, so averaging it within a group *is* that group's survival rate — a bar chart of the mean is the standard way to show this.

In [ ]:
# bar height = mean Survived per Sex group = survival rate for that group

sns.barplot(data=train, x="Sex", y="Survived")

plt.xticks([0, 1], ["Female", "Male"])  # relabel 0/1 back to readable text

plt.ylabel("Survival rate")

plt.title("Survival rate by Sex (train only)")

plt.show()

## Survival rate by Pclass



Same idea, grouped by ticket class instead of sex.

In [ ]:
sns.barplot(data=train, x="Pclass", y="Survived")

plt.ylabel("Survival rate")

plt.title("Survival rate by Pclass (train only)")

plt.show()

## Age distribution, split by Survived



Different chart type here — Age is continuous, not a category, so instead of one bar per group this overlays two histograms (died vs. survived) to see if certain age ranges skew one way.

In [ ]:
# hue="Survived" draws one histogram per group, overlaid on the same axes

sns.histplot(data=train, x="Age", hue="Survived", bins=30, kde=True, element="step")

plt.title("Age distribution by Survived (train only)")

plt.show()

## Fare distribution, split by Survived



A box plot instead of a histogram — good for comparing the spread and median of a number (Fare) across two groups (Survived) at a glance.

In [ ]:
sns.boxplot(data=train, x="Survived", y="Fare")

plt.title("Fare by Survived (train only)")

plt.show()

## Sex × Pclass interaction check



Everything above looked at one feature's relationship to `Survived` at a time. This checks something different: does Sex's *effect* on survival change depending on Pclass, instead of just adding independently? That's what an interaction is.



Fit with `statsmodels` instead of scikit-learn specifically because it reports p-values and standard errors — scikit-learn is built for prediction and doesn't. `C(Sex) * C(Pclass)` treats both as categories and auto-generates the interaction terms; look at the `:` rows in the summary — a significant p-value (< 0.05) there is real statistical evidence of an interaction, not just a visually bigger gap.

In [ ]:
import statsmodels.formula.api as smf



# text labels instead of raw numbers make the summary's row names self-explanatory

# (e.g. C(Sex)[T.Male] instead of C(Sex)[T.1]) — this is just for readability here,

# train/test themselves stay untouched

readable = train.assign(

    Sex=train["Sex"].map({0: "Female", 1: "Male"}),

    Pclass=train["Pclass"].map({1: "1st", 2: "2nd", 3: "3rd"}),

)



# C(...) tells statsmodels to treat Sex/Pclass as categories, not continuous numbers;

# "*" fits both main effects (Sex, Pclass) plus their interaction terms in one go

interaction_model = smf.logit("Survived ~ C(Sex) * C(Pclass)", data=readable).fit()

print(interaction_model.summary())

## What this analysis produced

Two conclusions came out of the above, and both are implemented in `preprocessing.ipynb` rather than here:

- **`Male_and_3rdClass`** — the only Sex × Pclass interaction that came back statistically significant (`Male:Pclass=2` did not), so it gets one targeted flag instead of a full Sex × Pclass cross.
- **One-hot encoded `Pclass`** — the effect isn't linear (1st ≈ 2nd, then a cliff at 3rd), so a raw 1/2/3 number would wrongly assume even spacing.

To test a change: edit `preprocessing.ipynb`, rename the config in `configs/` to describe the new feature set, then run `uv run train <config_name>`.